# 03 — Engenharia de Features: Embeddings Estáticos vs. Contextuais

Implementa a Fase 18 do plano de elaboração: compara embeddings estáticos
(FastText) e contextuais (BERTimbau) sobre o corpus de treino, além de uma
demonstração de seleção de features sobre a matriz TF-IDF usada pelos
modelos clássicos (`notebooks/04_ml_classico.ipynb`).

**Pré-requisito**: a etapa `features` já deve ter sido executada
(`uv run python src/main.py --stage features`), populando
`paths.training_corpus_file`.

**Embeddings estáticos (FastText)**: exigem um modelo pré-treinado local
(ex.: `cc.pt.300.bin`, várias centenas de MB, não versionado neste
repositório). Defina `FASTTEXT_MODEL_PATH` abaixo; a seção é pulada com
uma instrução clara caso o arquivo não exista, em vez de falhar.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

from config.paths import load_project_paths
from data.loader import load_training_example_dataset
from data.sampler import sample_stratified_subset
from visualization.theme import apply_project_theme, save_figure

apply_project_theme()
paths = load_project_paths()

# Amostra menor para manter a extração de embeddings/PCA rápida na exploração;
# a etapa `features` do pipeline processa o corpus completo.
training_corpus = load_training_example_dataset(paths.training_corpus_file)
sample_corpus = sample_stratified_subset(
    training_corpus, stratify_column="sentiment_label", sample_size=min(500, training_corpus.height)
)

FASTTEXT_MODEL_PATH = Path("cc.pt.300.bin")  # ajuste para o caminho local do modelo pré-treinado

## Embeddings contextuais (BERTimbau)

In [ ]:
from features.contextual_embeddings import extract_contextual_embeddings, load_contextual_encoder

contextual_encoder = load_contextual_encoder()
contextual_embeddings = extract_contextual_embeddings(sample_corpus, contextual_encoder)
contextual_embeddings.head()

## Embeddings estáticos (FastText), quando disponível localmente

In [ ]:
static_embeddings = None
if FASTTEXT_MODEL_PATH.exists():
    from features.static_embeddings import extract_static_embeddings, load_fasttext_model

    fasttext_model = load_fasttext_model(FASTTEXT_MODEL_PATH)
    static_embeddings = extract_static_embeddings(sample_corpus, fasttext_model)
    display(static_embeddings.head())
else:
    print(
        f"Modelo FastText não encontrado em '{FASTTEXT_MODEL_PATH}'. Baixe um "
        "modelo pré-treinado em português (ex.: "
        "https://fasttext.cc/docs/en/crawl-vectors.html) e ajuste "
        "FASTTEXT_MODEL_PATH acima para comparar com os embeddings contextuais."
    )

## Comparação estatística entre as matrizes de embedding

Esparsidade e norma L2 ajudam a antecipar o comportamento de cada
representação nos modelos a jusante (ex.: embeddings muito esparsos
favorecem modelos lineares; normas muito díspares entre amostras podem
exigir normalização antes do treino).

In [ ]:
from features.statistics import summarize_feature_matrix

feature_matrices = {"contextual_bertimbau": contextual_embeddings}
if static_embeddings is not None:
    feature_matrices["static_fasttext"] = static_embeddings

for matrix_name, feature_matrix in feature_matrices.items():
    summary = summarize_feature_matrix(feature_matrix, matrix_name=matrix_name)
    print(matrix_name, summary)

## Projeção 2D para inspeção visual

Usa PCA (`sklearn`, parte do stack principal do projeto) apenas para fins
de visualização — a redução de dimensionalidade usada de fato no pipeline
(`src/features/reduction.py`) é o autoencoder treinado, avaliado em
`notebooks/02_diagnostico_rotulagem.ipynb` pelo erro de reconstrução.

In [ ]:
from sklearn.decomposition import PCA

from visualization.embeddings import plot_embedding_scatter

embedding_columns = [column for column in contextual_embeddings.columns if column != "id"]
embedding_matrix = contextual_embeddings.select(embedding_columns).to_numpy()
coordinates_2d = PCA(n_components=2, random_state=42).fit_transform(embedding_matrix)

scatter_figure = plot_embedding_scatter(
    coordinates_2d,
    sample_corpus["sentiment_label"].to_list(),
    title="Projeção PCA 2D — Embeddings Contextuais (BERTimbau)",
)
save_figure(scatter_figure, "projecao_embeddings_contextuais", directory=paths.reports_figures_dir)

## Seleção de features sobre a matriz TF-IDF

Demonstra a redução de redundância (`select_features_by_redundancy`) e a
seleção pelas features mais correlacionadas com o alvo
(`select_k_best_features_by_target_correlation`) sobre a mesma matriz
TF-IDF usada pelos modelos clássicos.

In [ ]:
from constants.labels import transform_label_to_id
from features.lexical import compute_tfidf_features, pivot_tfidf_features_to_wide
from features.selection import (
    select_features_by_redundancy,
    select_k_best_features_by_target_correlation,
)

tfidf_long = compute_tfidf_features(sample_corpus)
tfidf_wide = pivot_tfidf_features_to_wide(tfidf_long)
n_features_before = len(tfidf_wide.columns) - 1

non_redundant = select_features_by_redundancy(tfidf_wide, correlation_threshold=0.95)
print(f"Features após remover redundância: {len(non_redundant.columns) - 1}/{n_features_before}")

target = [
    transform_label_to_id(label)
    for label in sample_corpus.join(tfidf_wide.select("id"), on="id")["sentiment_label"].to_list()
]
best_features = select_k_best_features_by_target_correlation(non_redundant, target, k=100)
print(
    f"Features após seleção pelas k mais correlacionadas ao alvo: {len(best_features.columns) - 1}"
)

## Conclusões

Registrar aqui: (1) diferenças de esparsidade/norma entre embeddings
estáticos e contextuais e sua implicação para a escolha de modelo
clássico (`notebooks/04_ml_classico.ipynb`) — modelos baseados em
distância/kernel (SVM RBF, k-NN) são mais sensíveis a normas díspares que
modelos lineares; (2) se a projeção 2D revela separação visível entre
classes já no espaço de embeddings, o que antecipa bom desempenho de
classificadores lineares simples; (3) quantas features TF-IDF sobrevivem
à seleção e se esse número é compatível com o tamanho do corpus de
treino (risco de overfitting em alta dimensionalidade).